# Comparing runs with 16 and 32 frames on WLASL 100 to 2000, with similar parameters

In [1]:
import json
from pathlib import Path
#locals
# from code.run_types import ResSet, RunRes
import pandas as pd
from run_types import ResSet, RunRes, GenInfo
from resulting import print_json

In [2]:
saicair = Path('results/saicair/')
runs_16_p = saicair / 'config_16f_15p.json'
runs_32_p = saicair / 'config_32f_15p.json'
assert runs_16_p.exists(), f"{runs_16_p} not found"
assert runs_32_p.exists(), f"{runs_32_p} not found"

with open(runs_16_p) as f:
    runs_16 = json.load(f)

with open(runs_32_p) as f:
    runs_32 = json.load(f)

## Parameters in common:

The only parameter that differs is the number of frames. In all cases, models trained on asl300 upward were initialised from the previous split. 

In [3]:
print_json(runs_16['spec'])

{
    "training": {
        "batch_size_equivalent": 8
    },
    "optimizer": {
        "eps": 1e-05,
        "backbone_init_lr": 0.0001,
        "backbone_weight_decay": 0.001,
        "classifier_init_lr": 0.001,
        "classifier_weight_decay": 0.001
    },
    "model_params": {
        "drop_p": 0.5
    },
    "data": {
        "num_frames": 16,
        "frame_size": 224,
        "train_augs": {
            "frame_size_strategy": "Random_crop",
            "normalise": true,
            "frame_sampler": {
                "method": "og"
            },
            "spatial_aug": [
                {
                    "type": "HORIZONTAL_FLIP",
                    "p": 0.5
                }
            ]
        },
        "test_augs": {
            "frame_size_strategy": "Centre_crop",
            "normalise": true,
            "frame_sampler": {
                "method": "og"
            }
        }
    },
    "early_stopping": {
        "metric": [
            "val",
          

## Runs 16 frames:

Lets check how many runs there are that match that spec:

In [4]:
for run in runs_16['results']:
    print_json(run['admin'])
    print()
print(len(runs_16['results']))

{
    "model": "MViTv2_S_16x4",
    "dataset": "WLASL",
    "split": "asl2000",
    "save_path": "runs/asl2000/MViTv2_S_16x4/exp000/checkpoints",
    "exp_no": "000",
    "recover": true,
    "config_path": "configfiles/asl2000/MViTv2_S_16x4/exp000.toml",
    "weight_path": "/home/luke/Code/SLR/code/runs/asl1000/MViTv2_S_16x4/exp000/checkpoints/best.pth"
}

{
    "model": "MViTv2_S_16x4",
    "dataset": "WLASL",
    "split": "asl1000",
    "save_path": "runs/asl1000/MViTv2_S_16x4/exp000/checkpoints",
    "exp_no": "000",
    "recover": false,
    "config_path": "configfiles/asl1000/MViTv2_S_16x4/exp000.toml",
    "weight_path": "/home/luke/Code/SLR/code/runs/asl300/MViTv2_S_16x4/exp000/checkpoints/best.pth"
}

{
    "model": "MViTv2_S_16x4",
    "dataset": "WLASL",
    "split": "asl300",
    "save_path": "runs/asl300/MViTv2_S_16x4/exp000/checkpoints",
    "exp_no": "000",
    "recover": false,
    "config_path": "configfiles/asl300/MViTv2_S_16x4/exp000.toml",
    "weight_path": "/home/

## Runs 32 frames:

Lets check how many runs there are that match that spec:

Note there were 3 extra asl100s. We will take exp007 because it is used as a starting point for the asl300 run


In [5]:
runs_32['results'] = runs_32['results'][:-3] 
for run in runs_32['results']:
    print_json(run['admin'])
    print()
print(len(runs_32['results']))

{
    "model": "MViTv2_B_32x3",
    "dataset": "WLASL",
    "split": "asl2000",
    "save_path": "runs/asl2000/MViTv2_B_32x3/exp000/checkpoints",
    "exp_no": "000",
    "recover": false,
    "config_path": "configfiles/asl2000/MViTv2_B_32x3/exp000.toml",
    "weight_path": "/home/luke/Code/SLR/code/runs/asl1000/MViTv2_B_32x3/exp000/checkpoints/best.pth"
}

{
    "model": "MViTv2_B_32x3",
    "dataset": "WLASL",
    "split": "asl1000",
    "save_path": "runs/asl1000/MViTv2_B_32x3/exp000/checkpoints",
    "exp_no": "000",
    "recover": false,
    "config_path": "configfiles/asl1000/MViTv2_B_32x3/exp000.toml",
    "weight_path": "/home/luke/Code/SLR/code/runs/asl300/MViTv2_B_32x3/exp000/checkpoints/best.pth"
}

{
    "model": "MViTv2_B_32x3",
    "dataset": "WLASL",
    "split": "asl300",
    "save_path": "runs/asl300/MViTv2_B_32x3/exp000/checkpoints",
    "exp_no": "000",
    "recover": false,
    "config_path": "configfiles/asl300/MViTv2_B_32x3/exp000.toml",
    "weight_path": "/home

## Now we can compare the runs

In [6]:
avail_acc_types = ["top_k_average_per_class_acc", "top_k_per_instance_acc"]
acc_type = avail_acc_types[1]
set_name = 'test'

In [7]:
df_format = []

for res in runs_16['results'] + runs_32['results']:
    df_format.append(
        {'model': res['admin']['model'], 'subset': res['admin']['split']} | {k : v for k, v in res['results'][set_name][acc_type].items()}
    )

df = pd.DataFrame(df_format)

In [8]:
df = df.rename(columns={"top1": "Top-1", "top5": "Top-5", "top10": "Top-10"})
df

,model,subset,Top-1,Top-5,Top-10
0,MViTv2_S_16x4,asl2000,0.399097,0.706148,0.796457
1,MViTv2_S_16x4,asl1000,0.511194,0.793710,0.861940
2,MViTv2_S_16x4,asl300,0.609281,0.880240,0.925150
3,MViTv2_S_16x4,asl100,0.736434,0.914729,0.961240
4,MViTv2_B_32x3,asl2000,0.445641,0.763460,0.837096
5,MViTv2_B_32x3,asl1000,0.567697,0.834755,0.896588
6,MViTv2_B_32x3,asl300,0.670659,0.887725,0.922156
7,MViTv2_B_32x3,asl100,0.775194,0.934109,0.965116


In [9]:
df['Top-1'] = df['Top-1'].apply(lambda x: f'{x*100:.2f}')
df['Top-5'] = df['Top-5'].apply(lambda x: f'{x*100:.2f}')
df['Top-10'] = df['Top-10'].apply(lambda x: f'{x*100:.2f}')

In [10]:
df

,model,subset,Top-1,Top-5,Top-10
0,MViTv2_S_16x4,asl2000,39.91,70.61,79.65
1,MViTv2_S_16x4,asl1000,51.12,79.37,86.19
2,MViTv2_S_16x4,asl300,60.93,88.02,92.51
3,MViTv2_S_16x4,asl100,73.64,91.47,96.12
4,MViTv2_B_32x3,asl2000,44.56,76.35,83.71
5,MViTv2_B_32x3,asl1000,56.77,83.48,89.66
6,MViTv2_B_32x3,asl300,67.07,88.77,92.22
7,MViTv2_B_32x3,asl100,77.52,93.41,96.51
